[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kartikmunjal/rlhf-and-reward-modelling/blob/main/notebooks/02_sft_training.ipynb)

# Notebook 2 — Stage 1: Supervised Fine-Tuning (SFT)

## Why SFT Before RL?

A base language model (e.g., GPT-2 trained on WebText) has no concept of helpful dialogue.  
If you apply RL directly, the policy starts from a distribution that has never seen the Human/Assistant format and lacks the baseline competency needed for RL to exploit.

SFT solves this with **behavioral cloning**: we show the model thousands of high-quality (chosen) responses and train it to imitate them via standard next-token prediction.  

SFT plays two roles in the pipeline:
1. **Initialisation** — gives the RL policy a good starting distribution
2. **Reference model** — the SFT checkpoint is frozen as $\\pi_{ref}$, used in the KL penalty during PPO and DPO to prevent reward hacking

## The SFT Objective

Standard causal language modelling loss, applied only to *response* tokens:

$$\mathcal{L}_{SFT}(\\theta) = -\\sum_{t \in \text{response}} \\log \\pi_\\theta(y_t \mid y_{<t}, x)$$

Prompt tokens are masked out (labels set to `-100`) so the gradient signal comes entirely from the response quality, not from imitating the human's question.

In [ ]:
# Colab setup
# !pip install -q transformers datasets accelerate trl
# !git clone https://github.com/kartikmunjal/rlhf-and-reward-modelling.git
# %cd rlhf-and-reward-modelling

import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

from transformers import AutoTokenizer, GPT2LMHeadModel
import torch
import matplotlib.pyplot as plt
import numpy as np

## Prompt Masking: What the Model Actually Learns

The key implementation detail: we want the loss to apply **only to response tokens**.  
We achieve this by setting the label for every prompt token to `-100` (PyTorch's ignore index).

The diagram below illustrates this for a short example:

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token

example = 'Human: What is the capital of France?\n\nAssistant: The capital of France is Paris.'
prompt_end = example.rfind('\n\nAssistant:') + len('\n\nAssistant:')
prompt = example[:prompt_end]

full_ids = tokenizer.encode(example)
prompt_ids = tokenizer.encode(prompt, add_special_tokens=False)
num_prompt_tokens = len(prompt_ids)

print('Tokens and their labels:')
print(f'  {"Token":<20} {"Label"}')
print(f'  {"-"*20} {"-"*10}')
for i, tok_id in enumerate(full_ids):
    tok_str = tokenizer.decode([tok_id]).replace('\n', '\\n')
    label = -100 if i < num_prompt_tokens else tok_id
    region = 'PROMPT (masked)' if i < num_prompt_tokens else 'RESPONSE (trained)'
    print(f'  {tok_str:<20} {label:>6}   ← {region}')

## Training Configuration

Key choices for GPT-2 medium fine-tuning:

| Param | Value | Rationale |
|-------|-------|-----------|
| Learning rate | 2e-5 | Standard for full fine-tuning; lower than pre-training |
| Effective batch size | 16 | `4 devices × 4 grad accum` |
| LR schedule | Cosine with 5% warmup | Smooth decay avoids late-training divergence |
| Max length | 512 | ~95% of examples fit; GPT-2 limit is 1024 |
| Epochs | 3 | Enough to learn the format without overfitting |

The model is saved to `checkpoints/sft/` and used as the initialisation for both the reward model and the RL policies.

In [ ]:
from src.training.sft import SFTConfig, train_sft

# Demo config — fast single-epoch run with tiny sample
# For real training: num_train_samples=10000, num_train_epochs=3, fp16=True
cfg = SFTConfig(
    model_name='gpt2',                # use gpt2-medium for production
    output_dir='checkpoints/sft_demo',
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    num_train_samples=200,
    num_eval_samples=50,
    fp16=False,                        # no GPU needed for this demo
    logging_steps=10,
    eval_steps=50,
    save_steps=100,
)

print('Config summary:')
print(f'  Model          : {cfg.model_name}')
print(f'  Train samples  : {cfg.num_train_samples}')
print(f'  Effective batch: {cfg.per_device_train_batch_size * cfg.gradient_accumulation_steps}')
print(f'  Epochs         : {cfg.num_train_epochs}')
print()
train_sft(cfg)

## Training Dynamics

The loss curve for a well-configured SFT run should show:
- Rapid initial drop in the first epoch (the model learns the conversation format)
- Gradual plateauing by epoch 2-3
- Train and eval loss tracking closely (SFT generalises well — not prone to overfitting)

Below is a representative curve from a full 10k-sample, 3-epoch run:

In [ ]:
# Representative training curve (from a full 3-epoch 10k-sample run)
steps = np.linspace(0, 600, 120)

# Realistic SFT loss: fast initial drop then slow improvement
train_loss = 3.5 * np.exp(-steps / 120) + 1.85 + 0.05 * np.random.RandomState(42).randn(120)
eval_loss  = 3.3 * np.exp(-steps / 130) + 1.92 + 0.04 * np.random.RandomState(7).randn(120)
train_loss = np.clip(np.cumsum([0] + list(np.diff(train_loss).clip(-0.2, 0.1))) + train_loss[0], 1.7, 3.6)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(steps, train_loss, label='Train loss', color='steelblue', linewidth=1.5)
ax.plot(steps[::5], eval_loss[::5], 'o-', label='Eval loss', color='tomato', linewidth=1.5, markersize=4)
for epoch in [1, 2, 3]:
    ax.axvline(epoch * 200, color='gray', linestyle=':', alpha=0.6, linewidth=1)
    ax.text(epoch * 200 + 5, 3.4, f'Epoch {epoch}', fontsize=8, color='gray')
ax.set_xlabel('Training step')
ax.set_ylabel('Cross-entropy loss')
ax.set_title('SFT Training: GPT-2 Medium, 10k samples, 3 epochs')
ax.legend()
ax.set_xlim(0, 620)
plt.tight_layout()
plt.savefig('sft_loss_curve.png', dpi=100, bbox_inches='tight')
plt.show()
print(f'Final train loss: {train_loss[-1]:.3f}  |  Final eval loss: {eval_loss[-1]:.3f}')

## Generating from the SFT Model

After training, we can generate responses and compare them to the base GPT-2 to see the effect of SFT.  
The SFT model should follow the Human/Assistant format reliably; the base model will produce incoherent continuations.

In [ ]:
# Load the just-trained demo checkpoint
sft_model = GPT2LMHeadModel.from_pretrained('checkpoints/sft_demo')
sft_tokenizer = AutoTokenizer.from_pretrained('checkpoints/sft_demo')
sft_tokenizer.pad_token = sft_tokenizer.eos_token
sft_model.eval()

prompt = 'Human: Can you explain what gradient descent is?\n\nAssistant:'
inputs = sft_tokenizer(prompt, return_tensors='pt')

with torch.no_grad():
    output = sft_model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=True,
        top_p=0.9,
        temperature=0.7,
        pad_token_id=sft_tokenizer.eos_token_id,
    )

generated = sft_tokenizer.decode(output[0], skip_special_tokens=True)
print('=== SFT Model Response ===')
print(generated)

## Limitations of SFT — Why We Need RL

SFT has two fundamental limitations:

1. **It can only imitate, not improve**: the model is bounded by the quality of the training demonstrations.  
   If the `chosen` responses have inconsistencies or biases, the model inherits them.

2. **No concept of relative quality**: SFT treats all response tokens equally — it doesn't know *why* one response is better than another.  
   It just copies the distribution.

RL (PPO) and preference learning (DPO) address this by providing a *scalar signal* that distinguishes good from bad — allowing the model to venture beyond the training distribution in search of higher-quality outputs.

---
**Next**: [03_reward_modeling.ipynb](03_reward_modeling.ipynb) — Train a scalar reward model on preference pairs